# Notebook 06 — Airflow ETL Pipeline Execution
**SuperStore Sales & Business Performance Analytics System**

This notebook orchestrates the full 9-task Airflow ETL pipeline in **two modes**:

| Mode | Description |
|------|-------------|
| **A — Local runner** | Runs all tasks in-process via `airflow tasks test`. No scheduler needed. Best for dev/CI. |
| **B — Full scheduler** | Starts Airflow webserver + scheduler, triggers DAG, polls until complete. |

### DAG: `superstore_etl_pipeline` (9 tasks)
```
validate_raw_data
      │
  ingest_data
      │
 clean_transform_data
     ├──────────────────────────┐
 run_pandas_analytics    run_spark_analytics
      │                         │
 generate_dashboard  ───────────┘
      │
 upload_to_azure   ← Blob + ADLS Gen2 + Databricks DBFS
      │
 data_quality_checks
      │
 notify_success
```
**Schedule**: `@daily`  **Retries**: 3 × 5 min delay  **Executor**: SequentialExecutor

## Step 0 — Environment Configuration

In [ ]:
import os, sys, subprocess, time
from pathlib import Path
from datetime import datetime

# ── Paths ─────────────────────────────────────────────────────────────────
PROJECT_ROOT  = Path().resolve().parent          # one level up from notebooks/
AIRFLOW_HOME  = PROJECT_ROOT / "airflow"
AIRFLOW_VENV  = AIRFLOW_HOME / ".venv"           # Python 3.12 venv with Airflow 2.9.3
AIRFLOW_BIN   = AIRFLOW_VENV / "bin" / "airflow"
VENV_PYTHON   = AIRFLOW_VENV / "bin" / "python"
DAG_ID        = "superstore_etl_pipeline"
EXECUTION_DATE = datetime.now().strftime("%Y-%m-%d")

# ── Airflow environment variables ─────────────────────────────────────────
AIRFLOW_ENV = {
    **os.environ,
    "AIRFLOW_HOME":                          str(AIRFLOW_HOME),
    "AIRFLOW__CORE__DAGS_FOLDER":            str(AIRFLOW_HOME / "dags"),
    "AIRFLOW__CORE__EXECUTOR":               "SequentialExecutor",
    "AIRFLOW__CORE__LOAD_EXAMPLES":          "False",
    "AIRFLOW__DATABASE__SQL_ALCHEMY_CONN":   f"sqlite:///{AIRFLOW_HOME}/airflow.db",
    "AIRFLOW__WEBSERVER__SECRET_KEY":        "superstore-airflow-key-2024",
    "PYTHONPATH":                            str(PROJECT_ROOT),
}

# ── Load .env credentials into AIRFLOW_ENV ────────────────────────────────
env_file = PROJECT_ROOT / ".env"
if env_file.exists():
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, _, v = line.partition("=")
            AIRFLOW_ENV[k.strip()] = v.strip().strip('"')

# ── Verify venv ───────────────────────────────────────────────────────────
assert AIRFLOW_BIN.exists(), (
    f"Airflow venv not found at {AIRFLOW_VENV}.\n"
    "Run in terminal:\n"
    "  python3.12 -m venv airflow/.venv\n"
    "  airflow/.venv/bin/pip install 'apache-airflow==2.9.3' "
    "--constraint https://raw.githubusercontent.com/apache/airflow/constraints-2.9.3/constraints-3.12.txt"
)

print(f"Project root  : {PROJECT_ROOT}")
print(f"Airflow home  : {AIRFLOW_HOME}")
print(f"Airflow venv  : {AIRFLOW_VENV}")
print(f"Execution date: {EXECUTION_DATE}")
print("Environment   : OK")

## Step 1 — Verify Airflow Version & DAG Registration

In [ ]:
def run_af(args, timeout=60):
    """Helper: run an airflow CLI command in the venv and return stdout."""
    result = subprocess.run(
        [str(AIRFLOW_BIN)] + args,
        env=AIRFLOW_ENV,
        capture_output=True, text=True, timeout=timeout
    )
    return result.stdout + result.stderr

# Airflow version
version_out = run_af(["version"])
print("Airflow version:", version_out.strip())

# Registered DAGs
dags_out = run_af(["dags", "list"])
print("\nRegistered DAGs:")
for line in dags_out.splitlines():
    if "superstore" in line or "dag_id" in line or "==" in line:
        print(" ", line)

In [ ]:
# Show DAG task dependency tree
tree_out = run_af(["tasks", "list", DAG_ID, "--tree"])
print(f"Task tree for '{DAG_ID}':")
for line in tree_out.splitlines():
    if "Task" in line or "<" in line:
        print(line)

---
## MODE A — Run All Tasks via `airflow tasks test`
Each task runs through the real Airflow task runner (dependency injection, XCom, logging)  
but **without** the scheduler — perfect for development and demonstration.

In [ ]:
import re

PIPELINE_TASKS = [
    "validate_raw_data",
    "ingest_data",
    "clean_transform_data",
    "run_pandas_analytics",
    "generate_dashboard",
    "upload_to_azure",
    "data_quality_checks",
    "notify_success",
]

# Patterns to surface from Airflow logs
IMPORTANT = re.compile(
    r"(rows|Ingested|Validation|Cleaning|Generated|Saved|Bronze|Silver|Gold|"
    r"Dashboard|DBFS|Blob|upload|quality|PASS|FAIL|ERROR|SUCCESS|KPI|Sales|Profit)",
    re.IGNORECASE
)

results = {}
print("=" * 65)
print(f"SUPERSTORE ETL — AIRFLOW TASK TEST ({EXECUTION_DATE})")
print("=" * 65)

for task_id in PIPELINE_TASKS:
    print(f"\n{'─'*65}")
    print(f"TASK: {task_id}")
    print(f"{'─'*65}")

    t0 = time.time()
    result = subprocess.run(
        [str(AIRFLOW_BIN), "tasks", "test", DAG_ID, task_id, EXECUTION_DATE],
        env=AIRFLOW_ENV,
        capture_output=True, text=True, timeout=600
    )
    elapsed = round(time.time() - t0, 1)
    success = result.returncode == 0

    # Surface relevant log lines
    output = result.stdout + result.stderr
    for line in output.splitlines():
        if IMPORTANT.search(line):
            # Strip ANSI colour codes
            clean = re.sub(r"\x1b\[[0-9;]*m", "", line).strip()
            print(f"  {clean}")

    status = "SUCCESS ✓" if success else "FAILED  ✗"
    print(f"\n  [{status}]  elapsed: {elapsed}s")
    results[task_id] = {"success": success, "elapsed": elapsed}

    if not success:
        # Print last 20 lines of output on failure
        print("  -- Last 20 lines of output --")
        for line in output.splitlines()[-20:]:
            print(" ", re.sub(r"\x1b\[[0-9;]*m", "", line))
        print("\nPipeline halted on first failure.")
        break

print("\n" + "=" * 65)
passed = sum(1 for v in results.values() if v["success"])
total  = len(results)
total_t = round(sum(v["elapsed"] for v in results.values()), 1)
print(f"Tasks passed : {passed}/{total}")
print(f"Total time   : {total_t}s")
print("=" * 65)

In [ ]:
# Task result summary table
import pandas as pd

rows = [
    {
        "Task": tid,
        "Status": "✓ SUCCESS" if info["success"] else "✗ FAILED",
        "Elapsed (s)": info["elapsed"],
    }
    for tid, info in results.items()
]
print(pd.DataFrame(rows).to_string(index=False))

---
## MODE B — Full Airflow Scheduler + Web UI
Starts the Airflow webserver and scheduler as daemons, triggers the DAG via CLI, then polls until complete.

> The DB and admin user are already initialised (done during project setup).  
> Web UI: **http://localhost:8080** — login: `admin` / `admin`

In [ ]:
# ── Start webserver and scheduler as background subprocesses ──────────────
log_dir = AIRFLOW_HOME / "logs"
log_dir.mkdir(exist_ok=True)

webserver_log = open(log_dir / "webserver.log", "a")
scheduler_log = open(log_dir / "scheduler.log", "a")

webserver_proc = subprocess.Popen(
    [str(AIRFLOW_BIN), "webserver", "--port", "8080"],
    env=AIRFLOW_ENV,
    stdout=webserver_log, stderr=webserver_log,
)
scheduler_proc = subprocess.Popen(
    [str(AIRFLOW_BIN), "scheduler"],
    env=AIRFLOW_ENV,
    stdout=scheduler_log, stderr=scheduler_log,
)

print(f"Webserver PID : {webserver_proc.pid}  (logs: airflow/logs/webserver.log)")
print(f"Scheduler PID : {scheduler_proc.pid}  (logs: airflow/logs/scheduler.log)")
print("\nWaiting 20s for services to start ...")
time.sleep(20)
print(f"Web UI: http://localhost:8080  (admin / admin)")

In [ ]:
# ── Trigger a manual DAG run ──────────────────────────────────────────────
RUN_ID = f"notebook_run_{datetime.now().strftime('%Y%m%dT%H%M%S')}"

trigger_out = run_af(["dags", "trigger", DAG_ID, "--run-id", RUN_ID])
print(trigger_out)
print(f"Triggered run_id: {RUN_ID}")

In [ ]:
# ── Poll until the DAG run completes ─────────────────────────────────────
MAX_WAIT = 900   # 15 minutes
INTERVAL = 20
elapsed  = 0
TERMINAL = {"success", "failed"}

print(f"Polling every {INTERVAL}s (max {MAX_WAIT}s) ...")

while elapsed < MAX_WAIT:
    raw = run_af(["dags", "state", DAG_ID, EXECUTION_DATE])
    state = raw.strip().splitlines()[-1].lower()
    print(f"  [{elapsed:>4}s]  DAG state: {state}")
    if any(s in state for s in TERMINAL):
        break
    time.sleep(INTERVAL)
    elapsed += INTERVAL
else:
    print(f"Timed out after {MAX_WAIT}s")

print(f"\nFinal DAG state: {state}")

In [ ]:
# ── Per-task status after scheduler run ───────────────────────────────────
all_tasks = [
    "validate_raw_data", "ingest_data", "clean_transform_data",
    "run_pandas_analytics", "run_spark_analytics",
    "generate_dashboard", "upload_to_azure",
    "data_quality_checks", "notify_success",
]

print(f"{'Task':<30}  State")
print("-" * 50)
for tid in all_tasks:
    raw = run_af(["tasks", "state", DAG_ID, tid, EXECUTION_DATE])
    task_state = raw.strip().splitlines()[-1]
    print(f"{tid:<30}  {task_state}")

In [ ]:
# ── Stop Airflow services ─────────────────────────────────────────────────
# Run this cell when you are done with the Airflow UI.
for proc, name in [(webserver_proc, "webserver"), (scheduler_proc, "scheduler")]:
    if proc.poll() is None:
        proc.terminate()
        print(f"Airflow {name} (PID {proc.pid}) stopped.")
    else:
        print(f"Airflow {name} already stopped.")

---
## Step 2 — Verify Pipeline Outputs

In [ ]:
# ── Local output files ────────────────────────────────────────────────────
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

output_dir = PROJECT_ROOT / "data" / "output"
csv_files  = sorted(output_dir.glob("*.csv"))
png_files  = sorted(output_dir.glob("*.png"))

print(f"Analytics CSVs ({len(csv_files)}):")
for f in csv_files:
    print(f"  {f.name:<42}  {f.stat().st_size/1024:>7.1f} KB")

print(f"\nDashboard PNGs ({len(png_files)}):")
for f in png_files:
    print(f"  {f.name:<42}  {f.stat().st_size/1024:>7.1f} KB")

In [ ]:
# ── Pipeline KPI Summary ──────────────────────────────────────────────────
import pandas as pd
from src.analytics.sales_analytics import SalesAnalytics
from src.utils.config import load_config
from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

cfg = load_config()
parquet_path = PROJECT_ROOT / cfg["paths"]["processed_data_dir"] / cfg["data"]["processed_file"]
df  = pd.read_parquet(parquet_path)
kpis = SalesAnalytics(df).kpi_summary()

print("Pipeline KPI Summary")
print("─" * 40)
print(f"  Total Sales     : ${kpis['total_sales']:>12,.2f}")
print(f"  Total Profit    : ${kpis['total_profit']:>12,.2f}")
print(f"  Profit Margin   :  {kpis['profit_margin_pct']:>11.2f}%")
print(f"  Total Orders    :  {kpis['total_orders']:>12,}")
print(f"  Avg Order Value : ${kpis['avg_order_value']:>12,.2f}")
print(f"  Loss-making SKUs:  {kpis['loss_items']:>12,}")
print(f"  Dataset rows    :  {len(df):>12,}")

In [ ]:
# ── Monthly Sales Trend (first 12 months) ─────────────────────────────────
monthly_csv = output_dir / "monthly_trend.csv"
if monthly_csv.exists():
    mdf = pd.read_csv(monthly_csv)
    print("Monthly Sales Trend (first 12 months):")
    print(mdf.head(12).to_string(index=False))
else:
    print("Run the pipeline first to generate monthly_trend.csv")

In [ ]:
# ── Azure Blob Storage contents ───────────────────────────────────────────
from src.storage.azure_blob import AzureBlobClient

blob   = AzureBlobClient()
blobs  = list(blob.container_client.list_blobs())
total_b = sum(b.size for b in blobs)

print(f"Blob Storage: {blob.container_name}  ({len(blobs)} objects  /  {total_b/1024/1024:.2f} MB)")
for b in blobs:
    print(f"  {b.name:<55}  {b.size/1024:>8.1f} KB")

In [ ]:
# ── ADLS Gen2 Medallion Architecture ─────────────────────────────────────
from src.storage.adls_client import ADLSClient

adls = ADLSClient()
print(f"ADLS account: {adls.account_name}   filesystem: {adls.filesystem}")
print()

for layer in ["bronze", "silver", "gold", "dashboard"]:
    paths = adls.list_paths(directory=layer)
    print(f"  {layer:<12}: {len(paths)} objects")
    for p in paths[:6]:
        print(f"    {p}")

print("\nabfss:// paths (for Databricks):")
for layer, uri in adls.get_databricks_paths().items():
    print(f"  {layer:<12}: {uri}")

---
## Step 3 — Re-run Individual Tasks
Use `airflow tasks test` to re-run any single task without touching upstream/downstream state.

In [ ]:
import re

TASK_TO_RERUN = "data_quality_checks"   # ← change to any task_id

print(f"Re-running task: {TASK_TO_RERUN} ...")
result = subprocess.run(
    [str(AIRFLOW_BIN), "tasks", "test", DAG_ID, TASK_TO_RERUN, EXECUTION_DATE],
    env=AIRFLOW_ENV,
    capture_output=True, text=True, timeout=300
)
output = result.stdout + result.stderr
# Show only meaningful lines
for line in output.splitlines():
    clean = re.sub(r"\x1b\[[0-9;]*m", "", line).strip()
    if any(kw in clean for kw in ["INFO", "WARN", "ERROR", "PASS", "FAIL", "rows"]):
        print(clean)
print(f"\nExit code: {result.returncode}")

---
## Summary

| Airflow Component | Value |
|-------------------|-------|
| DAG ID | `superstore_etl_pipeline` |
| Schedule | `@daily` |
| Tasks | 9 (validate → ingest → clean → pandas/spark → dashboard → azure → quality → notify) |
| Executor | `SequentialExecutor` (SQLite) |
| Airflow version | 2.9.3 (Python 3.12 venv) |
| DB | `airflow/airflow.db` |
| Web UI | http://localhost:8080 (admin/admin) — start Mode B |
| Venv | `airflow/.venv/bin/airflow tasks test <dag> <task> <date>` |

| Azure Output | Location |
|---|---|
| Blob Storage | `xcubeassignment/superstore-analytics` |
| ADLS Bronze | `sachinxcube/superstore/bronze/` |
| ADLS Silver | `sachinxcube/superstore/silver/` |
| ADLS Gold | `sachinxcube/superstore/gold/` |
| ADLS Dashboard | `sachinxcube/superstore/dashboard/` |
| Databricks DBFS | `/dbfs/superstore/` |

**Next step →** Open [05_azure_databricks_setup.ipynb](05_azure_databricks_setup.ipynb) in your Databricks workspace to mount ADLS and create Delta Lake tables.